# TEX 2025 잔차 -9 메커니즘 분석 + 정책 최적화 — 마스터 노트북 v6

> 두 목적의 통합 분석 (반사실 시뮬레이션)
> 1. **잔차 -9의 메커니즘 발견** — 왜 기대만큼 못했는가
> 2. **반사실 정책 시나리오** — 이렇게 했으면 +N승

## 작업 흐름

| Phase | 도구 | 산출 |
|---|---|---|
| 1-5 | Markov + bullpen state augmentation | 시뮬 baseline (잔차 -6 자연 재현) |
| 6 | leverage penalty + 이닝 중 closer 교체 | 매치업 정밀화 |
| 7' | 시기별 closer 풀 (Jackson 7/23 DFA / Maton 8/1 합류) | 트레이드 반영 |
| 8 | NSGA-II 12차원 σ Pareto | 정책 trade-off 표면 + archetype 3종 |
| 옵션 A | 무승부 50% W 분배 | 동점 처리 (이전 동점=패의 보정) |

## 모듈 구조

- `simulator.py` — 타자 markov + LineupPool (`team_boosts` 인터페이스)
- `markov_pitching.py` — 투수 markov + bullpen state (`pitcher_adjustments` 인터페이스)
- `integrated_sim.py` — 두 시뮬 wrapper + 연장전 옵션 A
- `nsga_search.py` — σ 12차원 NSGA-II + Pareto 자동 분석

In [1]:
import sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 한국어 폰트
for font in ['AppleGothic', 'NanumGothic', 'Apple SD Gothic Neo']:
    if any(font in f.name for f in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = font
        break
plt.rcParams['axes.unicode_minus'] = False

# Phase 8 모듈 import
from integrated_sim import run_integrated_simulation, _ensure_loaded
from nsga_search import run_nsga_parallel, select_archetypes, save_pareto, SIGMA_DIMS, DIM_IDS
from markov_pitching import simulate_tex_RA
from simulator import LineupPool

print('모듈 import OK')
print()
print(f'σ 12차원 (총 {len(DIM_IDS)}개):')
for i, dim in enumerate(SIGMA_DIMS):
    print(f'  {i:2d}. {dim["id"]:10s}  ({dim["group"]}.{dim["sub"]}.{dim["key"]})  ∈ [{dim["lo"]:.2f}, {dim["hi"]:.2f}]')

# 환경 검증 — 1시즌
_ensure_loaded()
df_test = run_integrated_simulation(n_seasons=1, seeds=[0])
print()
print(f'1시즌 검증: W={df_test["W"].iloc[0]}, RS={df_test["RS"].iloc[0]}, RA={df_test["RA"].iloc[0]}')

모듈 import OK

σ 12차원 (총 12개):
   0. h_hr        (hitter.team.hr_mult)  ∈ [0.85, 1.30]
   1. h_bb        (hitter.team.bb_mult)  ∈ [0.85, 1.30]
   2. h_k         (hitter.team.k_mult)  ∈ [0.70, 1.15]
   3. h_single    (hitter.team.single_mult)  ∈ [0.85, 1.30]
   4. p_cl_K      (pitcher.closer.K_pct)  ∈ [0.80, 1.25]
   5. p_cl_BB     (pitcher.closer.BB_pct)  ∈ [0.75, 1.20]
   6. p_cl_BAB    (pitcher.closer.BABIP)  ∈ [0.80, 1.20]
   7. p_cl_HR     (pitcher.closer.HR_pct)  ∈ [0.50, 1.30]
   8. p_su_K      (pitcher.setup.K_pct)  ∈ [0.80, 1.25]
   9. p_st_HR     (pitcher.starter.HR_pct)  ∈ [0.50, 1.30]
  10. c_K         (pitcher.common.team_K_pct_mult)  ∈ [0.90, 1.15]
  11. c_BB        (pitcher.common.team_BB_pct_mult)  ∈ [0.90, 1.15]


/Users/jisoyun/Desktop/Final_Baseball/Final/markov_pitching.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pxp['tex_pitching'] = (



1시즌 검증: W=79, RS=651, RA=660


## Phase 1-7' 시뮬 메커니즘 (잔차 -9의 -6 자연 재현)

### 핵심 결과

TEX 2025 — 실제 W=81, RS=684, RA=605 → 피타고리안 기대 W ≈ 90.06 → **잔차 -9.06**

### 시뮬에 박힌 메커니즘들

| 메커니즘 | Phase | 효과 |
|---|---|---|
| 24 base-out × pitcher_tier × inning Markov | 1A-1B | 이닝별 상황별 시뮬 |
| Pitch-by-Pitch count progression | 5 | 카운트 분포 효과 |
| 1점차 leverage penalty (closer ΔK -0.116) | 6C | high-leverage 약화 정량화 |
| 위기 시 closer 조기 등판 | 6B | 매치업 정확 |
| 시기별 closer 풀 (Jackson 7/23 / Maton 8/1) | 7' | 트레이드 반영 |
| 무승부 50% W 분배 | 옵션 A | 연장전 단순 처리 |

### 메커니즘이 만든 효과

다음 셀에서 baseline 시뮬 162×20시즌을 돌려 *잔차 -9의 어느 정도가 메커니즘으로 자연 재현*되는지 측정합니다.

In [2]:
print('Phase 6-7\' baseline 시뮬 (162×20 시즌)...')
t0 = time.time()
df_baseline = run_integrated_simulation(n_seasons=20)
elapsed = time.time() - t0
print(f'완료 ({elapsed:.1f}s)')
print()
print('=== baseline 시뮬 결과 ===')
print(f'  W : mean={df_baseline["W"].mean():.2f}  std={df_baseline["W"].std():.2f}')
print(f'  RS: mean={df_baseline["RS"].mean():.1f}  std={df_baseline["RS"].std():.1f}')
print(f'  RA: mean={df_baseline["RA"].mean():.1f}  std={df_baseline["RA"].std():.1f}')
print()
print('=== 잔차 분석 ===')
print(f'  실제 W=81 대비:        {df_baseline["W"].mean() - 81:+.2f}승  (시뮬이 over/under-estimate)')
print(f'  피타고리안 W≈90 대비: {df_baseline["W"].mean() - 90.06:+.2f}승  ← 시뮬 잔차')
print(f'  실제 잔차: -9.06  →  시뮬 메커니즘이 자연 재현한 양: 약 {abs(df_baseline["W"].mean() - 90.06):.1f}승')

baseline_W = df_baseline['W'].mean()

Phase 6-7' baseline 시뮬 (162×20 시즌)...
완료 (34.9s)

=== baseline 시뮬 결과 ===
  W : mean=83.95  std=6.89
  RS: mean=670.5  std=36.8
  RA: mean=650.0  std=26.6

=== 잔차 분석 ===
  실제 W=81 대비:        +2.95승  (시뮬이 over/under-estimate)
  피타고리안 W≈90 대비: -6.11승  ← 시뮬 잔차
  실제 잔차: -9.06  →  시뮬 메커니즘이 자연 재현한 양: 약 6.1승


## Phase 8 — NSGA-II Pareto 탐색 (정통 다목적 최적화)

### 12차원 σ 정의 (B-결과 3 재설계)

velocity/spin은 PXP within-pitcher LPM 회귀에서 인과 식별 실패(Phase 8B 참조) → σ에서 제거. 대신 `team_K_pct_mult` / `team_BB_pct_mult` 같은 의미 명확한 multiplier로 교체.

| 그룹 | 차원 | 범위 | 의미 |
|---|---|---|---|
| 타자 4 | hr_mult, bb_mult, k_mult, single_mult | [0.85, 1.30], k는 [0.70, 1.15] | team-wide multiplier |
| 투수 6 | closer K_pct/BB_pct/BABIP/HR_pct, setup K_pct, starter HR_pct | [0.80, 1.25], HR은 [0.50, 1.30] | tier multiplier |
| 공통 2 | team_K_pct_mult, team_BB_pct_mult | [0.90, 1.15] | 모든 tier 일괄 |

### 3-objective

- **max W ↑** — 162경기 평균 승수
- **min σ_norm ↓** — RMS|x-1|, 정책 변경 비용 (1.0에서 떨어진 정도의 RMS)
- **min RA ↓** — 162경기 평균 실점

### 알고리즘

- pymoo 0.6.1.6 NSGA-II
- LHS 초기화, SBX crossover (η=15), PM mutation (η=20), eliminate_duplicates=True
- pop=50, gen=15, n_seasons=20 → 750 evals × 20시즌 = 15,000시즌
- 6 worker spawn 병렬화 → 약 60분

NSGA-II 자체 실행은 별도 호출 (시간 소요). 본 노트북은 결과 CSV(`nsga_pareto_phase8_optA_*.csv`)를 로드해서 분석합니다.

In [ ]:
# Phase 8 결과 로드 — v3(이닝 단위 인터리빙 + 책임 주자 ER) 우선, optA → 일반 fallback
csv_files = sorted(Path('.').glob('nsga_pareto_phase8_v3_*.csv'))
if not csv_files:
    csv_files = sorted(Path('.').glob('nsga_pareto_phase8_optA_*.csv'))
    if csv_files:
        print(f'⚠ v3 결과 없음 — optA 결과 사용: {csv_files[-1].name}')
        print('   (이닝 단위 시뮬 + 책임 주자 ER은 결과에 미반영)')
if not csv_files:
    csv_files = sorted(Path('.').glob('nsga_pareto_phase8_*.csv'))
    if csv_files:
        print(f'⚠ optA 결과도 없음 — 이전 NSGA-II 결과 사용: {csv_files[-1].name}')
phase8 = pd.read_csv(csv_files[-1])
print(f'Pareto CSV: {csv_files[-1].name}, 점수: {len(phase8)}')

DIMS = ['h_hr','h_bb','h_k','h_single','p_cl_K','p_cl_BB','p_cl_BAB','p_cl_HR','p_su_K','p_st_HR','c_K','c_BB']

# 분포
print()
print('=== Pareto 분포 ===')
print(f'  W      : [{phase8["W"].min():.1f}, {phase8["W"].max():.1f}]')
print(f'  σ_norm : [{phase8["sigma_norm"].min():.3f}, {phase8["sigma_norm"].max():.3f}]')
print(f'  RA     : [{phase8["RA"].min():.1f}, {phase8["RA"].max():.1f}]')

# 권장 zone
small = phase8[phase8['sigma_norm'] <= 0.10]
n_recover = ((phase8['sigma_norm']<=0.10) & (phase8['W']>=81)).sum()
n_plus4   = ((phase8['sigma_norm']<=0.10) & (phase8['W']>=85)).sum()
print()
print(f'=== 현실 권장 zone (σ_norm ≤ 0.10) — {len(small)}점 ===')
print(f'  잔차 만회 (W≥81): {n_recover}점,   +4승 향상 (W≥85): {n_plus4}점')
print()
print('최소 정책 변경 후보 (σ 작은 순 5점):')
print(small.sort_values('sigma_norm').head(5)[['W','sigma_norm','RA']+DIMS[:6]].to_string(index=False, float_format='%.3f'))

# Archetype
print()
print('=== Archetype 3종 ===')
arch = phase8[phase8['archetype'].isin(['aggressive','balanced','conservative'])]
print(arch[['archetype','W','sigma_norm','RA']+DIMS].to_string(index=False, float_format='%.3f'))

# Driver
print()
print('=== 차원별 W 상관 (driver 순위) ===')
corr = pd.Series({d: phase8[d].corr(phase8['W']) for d in DIMS}).sort_values()
for d, r in corr.items():
    bar = '█' * int(abs(r) * 20)
    print(f'  {d:10s}  r={r:+.3f}  {"-" if r<0 else "+"}{bar}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# σ vs W (현실 권장 zone 강조 + 기준선들)
ax = axes[0]
sc = ax.scatter(phase8['sigma_norm'], phase8['W'], c=phase8['RA'], cmap='RdYlGn_r',
                s=80, alpha=0.85, edgecolors='k', linewidth=0.5)
ax.axhline(81,    color='red',   ls='--', lw=1.2, label='실제 W=81')
ax.axhline(90.06, color='black', ls=':',  lw=1.2, label='피타고리안 ≈ 90.06')
ax.axhline(baseline_W, color='blue', ls='--', lw=1.2,
           label=f'시뮬 baseline {baseline_W:.1f}')
ax.axvspan(0, 0.10, alpha=0.1, color='green', label='현실 권장 zone (σ≤10%)')

arch_colors = {'aggressive':'red', 'balanced':'orange', 'conservative':'blue'}
for _, row in arch.iterrows():
    a = row['archetype']
    ax.scatter([row['sigma_norm']], [row['W']], c=arch_colors[a], s=240,
               marker='*', edgecolors='k', linewidth=1.5, zorder=5)
    ax.annotate(f' {a}', (row['sigma_norm'], row['W']), fontsize=9, weight='bold')
ax.set_xlabel('σ_norm (정책 변경 비용)')
ax.set_ylabel('W (162경기 평균 승수)')
ax.set_title('Phase 8 Pareto front — 정책 비용 vs 승수')
ax.grid(alpha=0.3)
ax.legend(loc='lower right', fontsize=9)
fig.colorbar(sc, ax=ax, shrink=0.7, label='RA')

# Driver
ax = axes[1]
colors = ['#d62728' if v < 0 else '#2ca02c' for v in corr.values]
ax.barh(corr.index, corr.values, color=colors, alpha=0.85, edgecolor='k')
ax.axvline(0, color='k', lw=0.8)
ax.set_xlabel('Pearson r (vs W)')
ax.set_title('차원별 W 상관 — driver 순위')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## v5 ML Pareto와 cross-check

v5 노트북 `5.통합본(0428+ML 수정)_v5.ipynb`의 Phase 1D NSGA-II:
- **6차원 stats** (sv_pct, ir_pct, onerun_wp, xi_wp, HR9, BB9)
- **ML 잔차 모델 평가자** (ridge/lasso/RF/XGB ensemble로 잔차 예측)
- 결과: delta = 잔차 만회 +2.9 ~ +4.9승

Phase 8 (지금):
- **12차원** (타자 4 + 투수 6 + 공통 2)
- **시뮬 평가자** (162경기 시뮬 직접)
- 결과: W +0 ~ +30+ (절대값)

### 두 결론의 비교

| 비교 항목 | v5 ML (6차원) | Phase 8 시뮬 (12차원) | 의미 |
|---|---|---|---|
| W 향상 범위 | +2.9 ~ +4.9 | +0 ~ +30+ | Phase 8이 ML extrapolation 한계 우회 |
| **HR9 / starter HR%** | -0.27 (큰 driver) | -48% (큰 driver) | **✓ 핵심 driver 일치** |
| BB9 / closer BB% | -0.58 | 약한 driver | 부분 일치 |
| 타자 영역 | **없음** | h_single +24%, h_k -25% | **Phase 8 신규 발견** |
| sv_pct/onerun_wp/ir_pct/xi_wp | 4개 차원 | (시뮬 leverage penalty로 흡수) | Phase 8 baseline에 이미 포함 |

### 핵심 인사이트

1. **양쪽 일치하는 driver — robust signal**: starter HR%↓
2. **Phase 8 신규 발견**: 타자 단타·K가 잔차 만회의 *가장 큰* driver — v5 ML이 *없던 차원*이라 발견 불가능했음
3. **v5 ML +5승 한계**: 학습 데이터 분포 밖 extrapolation 보수적. 시뮬은 메커니즘 기반이라 그 한계 없음
4. **두 도구가 *다른 질문*에 답함**:
   - ML: 잔차 -9의 *원인* 분석 (TEX 타자가 평범했으니 타자가 원인 아님)
   - 시뮬: 잔차 -9를 *만회/초과*하는 *반사실 정책* (타자 강화가 가장 효율적)
   - 모순이 아니라 상보적

In [ ]:
# v5 결과 (수동 입력 — output/pareto_summary.csv 참조)
v5_arch = pd.DataFrame([
    {'archetype':'공격적', 'delta':+4.889, 'std':0.6524, 'top_driver':'BB9 -0.58, HR9 -0.27, xi_wp +0.22'},
    {'archetype':'균형점', 'delta':+3.720, 'std':0.2047, 'top_driver':'BB9 -0.56, HR9 -0.27, xi_wp +0.22'},
    {'archetype':'보수적', 'delta':+2.919, 'std':0.0019, 'top_driver':'onerun_wp +0.11, xi_wp +0.18'},
])
print('=== v5 ML Pareto archetype (delta = 잔차 만회량, σ 단위 = 표준편차) ===')
print(v5_arch.to_string(index=False))
print()

print('=== Phase 8 시뮬 Pareto archetype (W = 절대값) ===')
print(arch[['archetype','W','sigma_norm','RA']].to_string(index=False, float_format='%.3f'))
print()

print('=== 매핑 표 (driver 일치 여부) ===')
print(f'{"v5 차원":18s}  {"Phase 8 대응":28s}  일치 여부')
print('-' * 75)
for v5_dim, p8_dim, match in [
    ('HR9 ↓ (피홈런)',  'p_st_HR + p_cl_HR ↓',  '✓ 일치 (방향+크기)'),
    ('BB9 ↓ (볼넷)',    'p_cl_BB + c_BB',       '부분 일치'),
    ('sv_pct ↑',         '(시뮬 leverage 박힘)',  '시뮬 baseline'),
    ('onerun_wp ↑',     '(시뮬 leverage 박힘)',  '시뮬 baseline'),
    ('xi_wp / ir_pct',   '(시뮬 미반영)',          '— (다른 도메인)'),
    ('(없음)',           'h_hr/h_bb/h_k/h_single', 'Phase 8 신규'),
    ('(없음)',           'c_K (team K_pct mult)',  'Phase 8 신규'),
]:
    print(f'{v5_dim:18s}  {p8_dim:28s}  {match}')

## 결론

### 두 목적의 통합 답

**(1) 왜 기대만큼 못했는가** — 잔차 -9의 메커니즘
- 시뮬 baseline이 잔차 -9 중 약 **70%(-6)를 자연 재현** → 메커니즘 가설 *consistency check* 통과
- 핵심 메커니즘:
  - high-leverage 상황 stat 약화 (closer ΔK -0.116)
  - 1점차 매치업 약점 (Phase 4: Garcia BABIP 0.625)
  - starter HR% 부진
  - 시기별 closer 풀 (Jackson 7/23 ~ Maton 8/1 공백)
- 잔차 -9 중 -6 정도가 "발견된 원인"이고 나머지 -3은 운/미발견 메커니즘

**(2) 이렇게 했으면 +N승** — 정책 시나리오
- 작은 정책 변경(σ_norm ≤ 0.10)으로도 잔차 -9 만회 가능
- 핵심 driver:
  1. 타자 단타 증가 (h_single, r_W ≈ +0.91)
  2. 타자 K 감소 (h_k, r_W ≈ -0.84)
  3. starter HR% 감소 (p_st_HR, r_W ≈ -0.80)
  4. team K% 증가 (c_K, r_W ≈ +0.59)
- v5 ML과 cross-check: starter HR%↓ 일치 (robust), 타자 영역은 Phase 8 신규 발견

### 발표용 핵심 메시지

> "TEX 2025 잔차 -9 중 -6은 시뮬 메커니즘으로 자연 재현되어 *발견된 원인*이 됐고, 나머지 -3 + 추가 향상은 σ 정책으로 만회 가능. 정책의 가장 큰 driver는 타자 단타와 starter HR — 그 중 starter HR은 ML 잔차 분석과도 일치 (robust)."

### 한계 (정직 평가)

- **단일 팀(TEX 2025) 분석** — 다른 팀 cross-validation 미수행 → 외적 타당성 미검증
- **인과 증명 부재** — 시뮬 메커니즘이 잔차 -6 만든다는 것은 *consistency check*이지 *증명* X. DML/IV 같은 인과추론 미사용
- **σ_norm 균등 가중** — 모든 차원 동등 weight. 실제 정책 비용은 비대칭 (단타 +5% < hr +5% < closer 강화)
- **큰 σ 영역 신뢰 한계** — 타자 hr +20% 같은 시나리오는 학습 데이터 분포 밖 반사실 추정. 시뮬 메커니즘 가정 하에서만 유효
- **PXP 회귀 power 부족** — 5,678 PA로 within-pitcher velocity 효과 식별 실패

학부 캡스톤 / 학회 워크숍 포스터로는 강함, 학술지 단독 투고는 추가 작업 필요.

---

*작성일: 2026-05-09*
*도구: Markov chain Monte Carlo + ML 잔차 + NSGA-II (pymoo) + PXP 정밀 시뮬 + multiprocessing 병렬화*